# Phase 2 — Condition 6: Forced Utilization (Asymmetric Dropout + Auxiliary Loss)

**Goal:** Overcome the modality competition that causes the clinical branches to dominate early training. We will force the gated fusion head to utilize the highly discriminative features from our custom MAE self-supervised encoder (from Condition 4/5).

**Methods Implemented:**
1. **Asymmetric Modality Dropout:** We apply aggressive dropout ($p=0.5$) *only* to the clinical and biomarker branches during training. The MRI branch is never dropped ($p=0.0$). This blinds the network to the "easy" features half the time, forcing it to optimize the MRI pathway.
2. **Auxiliary Unimodal Loss:** We attach a temporary classification head directly to the output of the 3D Swin MRI encoder. During training, we calculate a separate classification loss for the MRI branch alone and add it to the total loss. This ensures the Swin encoder receives strong gradient signals even if the fusion gate initially assigns it a low weight.

**Target to Beat:**
* Condition 3b (Frozen CT): MRI Gate = 0.141
* Condition 5 (Improved Gate): MRI Gate = 0.1308

Cell 2: Code (Architecture Modifications)

In [ ]:
from google.colab import drive
drive.mount('<DRIVE_ROOT>', force_remount=True)
!pip install -q "monai==1.5.0" nibabel torchio imbalanced-learn scikit-learn
import os; os._exit(0)


Mounted at <DRIVE_ROOT>
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.6/203.6 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 130.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

Cell 2: Config & Paths

In [15]:
import os, json, numpy as np, torch, datetime
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torchio as tio, pandas as pd
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score, matthews_corrcoef, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from monai.networks.nets import SwinUNETR
device=torch.device('cuda')

# ===== SET THIS =====
ENCODER_TO_EVAL='mae'   # Using the best SSL encoder
# ====================
CONDITION_NAME = f'condition6b_gentle_forced_{ENCODER_TO_EVAL}'

DRIVE='<DRIVE_ROOT>/ADNI_NewDS/'; RESULTS=os.path.join(DRIVE,'results')
BRAIN_DIR=os.path.join(RESULTS,'processed_mri_scans_brain')
SPLITS=os.path.join(RESULTS,'patient_id_splits.npz')
CSV=os.path.join(RESULTS,'project_data_cleaned.csv')
BIO=os.path.join(RESULTS,'preprocessed_biomarker_sequences.npy')
PHASE2=os.path.join(DRIVE,'Phase2_Collapse_Study')
SSL_DIR=os.path.join(PHASE2,'ssl_pretrain')
SSL_ENCODER_PATH=os.path.join(SSL_DIR, f'ssl_{ENCODER_TO_EVAL}_encoder.pth')
COND_DIR=os.path.join(PHASE2,'results',CONDITION_NAME); os.makedirs(COND_DIR, exist_ok=True)
LAB_LOG=os.path.join(PHASE2,'phase2_lab_log.jsonl')

IMG_SIZE=(96,96,96); FEATURE_SIZE=48; VIT_DIM=768; NUM_CLASSES=3; SEED=42
torch.manual_seed(SEED); np.random.seed(SEED)
assert os.path.exists(SSL_ENCODER_PATH), f'Missing SSL encoder: {SSL_ENCODER_PATH}'
print('Evaluating:', CONDITION_NAME)

Evaluating: condition6b_gentle_forced_mae


Cell 3: Data & Loaders

In [16]:
sp=np.load(SPLITS,allow_pickle=True)
pids_tr,pids_va,pids_te=sp['pids_train'],sp['pids_val'],sp['pids_test']
y_tr,y_va,y_te=sp['labels_train'],sp['labels_val'],sp['labels_test']
clin_df=pd.read_csv(CSV)
FEATS=['AGE','PTGENDER','PTEDUCAT','APOE4','MMSE','ADAS13','RAVLT_immediate','RAVLT_learning','RAVLT_forgetting','FAQ']
patient_clin={p:torch.tensor(g[FEATS].values,dtype=torch.float32) for p,g in clin_df.groupby('PTID')}
bio_raw=np.load(BIO,allow_pickle=True); bio_obj=bio_raw.item() if (hasattr(bio_raw,'dtype') and bio_raw.dtype==object and bio_raw.ndim==0) else bio_raw
patient_bio={k:torch.tensor(np.asarray(v),dtype=torch.float32) for k,v in bio_obj.items()}
CLIN_DIM=len(FEATS); BIO_DIM=next(iter(patient_bio.values())).shape[-1]

etf=tio.Compose([tio.Resize(IMG_SIZE), tio.ZNormalization(masking_method=tio.ZNormalization.mean)])
ttf=tio.Compose([tio.RandomFlip(axes='LR'), tio.Resize(IMG_SIZE), tio.ZNormalization(masking_method=tio.ZNormalization.mean)])
class DS(Dataset):
    def __init__(s,pids,labels,tf): s.pids=list(pids); s.y=torch.tensor(labels,dtype=torch.long); s.tf=tf
    def __len__(s): return len(s.pids)
    def __getitem__(s,i):
        p=s.pids[i]; vol=np.load(os.path.join(BRAIN_DIR,f'{p}_processed.npy'))
        subj=tio.Subject(mri=tio.ScalarImage(tensor=torch.tensor(vol,dtype=torch.float32).unsqueeze(0)))
        return s.tf(subj).mri.tensor.squeeze(0), patient_clin[p], patient_bio[p], s.y[i]
def coll(b):
    m,c,bi,y=zip(*b); return torch.stack(m),pad_sequence(c,batch_first=True),pad_sequence(bi,batch_first=True),torch.stack(y)

# PER INSTRUCTION: Increased batch size to 8
BATCH_SIZE=8
tr=DataLoader(DS(pids_tr,y_tr,ttf),batch_size=BATCH_SIZE,shuffle=True,collate_fn=coll,num_workers=2)
va=DataLoader(DS(pids_va,y_va,etf),batch_size=BATCH_SIZE,shuffle=False,collate_fn=coll,num_workers=2)
te=DataLoader(DS(pids_te,y_te,etf),batch_size=BATCH_SIZE,shuffle=False,collate_fn=coll,num_workers=2)
print(f'data ready. Batches: TR={len(tr)}, VA={len(va)}, TE={len(te)}')

data ready. Batches: TR=17, VA=4, TE=4


Cell 4: Load SSL Encoder

In [17]:
class SwinEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.swin=SwinUNETR(in_channels=1,out_channels=1,feature_size=FEATURE_SIZE,use_checkpoint=True).swinViT
        self.pool=nn.AdaptiveAvgPool3d(1)
    def forward(self,x): return self.pool(self.swin(x)[-1]).flatten(1)

encoder=SwinEncoder()
sd=torch.load(SSL_ENCODER_PATH, map_location='cpu')
missing,unexpected=encoder.load_state_dict(sd, strict=False)
matched=len(encoder.state_dict())-len(missing)
frac=matched/len(encoder.state_dict())
print(f'Loaded {frac:.1%} of SSL encoder keys')
assert frac>=0.8, 'SSL encoder did not load cleanly.'
encoder=encoder.to(device)
print('Your SSL encoder loaded. NOT CT-pretrained, NOT random.')

Loaded 100.0% of SSL encoder keys
Your SSL encoder loaded. NOT CT-pretrained, NOT random.


Cell 5: Architecture (Asym Dropout + Aux Loss) & Probe

In [18]:
class LSTMBranch(nn.Module):
    def __init__(s,ind,h=128,o=128):
        # FIXED: Set internal LSTM dropout back to a standard 0.2 so it can learn properly
        super().__init__(); s.l=nn.LSTM(ind,h,2,batch_first=True,dropout=0.2); s.f=nn.Linear(h,o); s.r=nn.ReLU()
    def forward(s,x): _,(h,_)=s.l(x); return s.r(s.f(h[-1]))

class TriModalForcedGate(nn.Module):
    # Default set to 0.2 here
    def __init__(s,enc,cd,bd,nc=3,mod_dropout_clin=0.2):
        super().__init__(); s.encoder=enc; s.mod_dropout_clin=mod_dropout_clin
        for p in s.encoder.parameters(): p.requires_grad=False
        s.proj=nn.Sequential(nn.Linear(VIT_DIM,256),nn.ReLU(),nn.Linear(256,128))
        s.cb=LSTMBranch(cd); s.bb=LSTMBranch(bd)

        # Per-modality LayerNorm from Cond 5
        s.norm_img=nn.LayerNorm(128); s.norm_clin=nn.LayerNorm(128); s.norm_bio=nn.LayerNorm(128)

        # Gate + learned temp from Cond 5
        s.gate_lin=nn.Sequential(nn.Linear(384,128),nn.ReLU(),nn.Linear(128,3))
        s.log_temp=nn.Parameter(torch.zeros(1))

        s.clf=nn.Sequential(nn.LayerNorm(128),nn.Linear(128,64),nn.ReLU(),nn.Dropout(0.5),nn.Linear(64,nc))

        # NEW: Auxiliary Classifier for MRI only
        s.aux_mri=nn.Sequential(nn.LayerNorm(128),nn.Linear(128,64),nn.ReLU(),nn.Linear(64,nc))

    def imgfeat(s,m):
        with torch.no_grad(): f=s.encoder(m.unsqueeze(1))
        return s.proj(f)

    def forward(s,m,c,b,rg=False,return_aux=False):
        fi=s.norm_img(s.imgfeat(m)); fc=s.norm_clin(s.cb(c)); fb=s.norm_bio(s.bb(b))

        # NEW: Asymmetric Modality Dropout - drop clin/bio heavily (p=0.2), NEVER drop MRI
        if s.training and s.mod_dropout_clin>0:
            r=torch.rand(2)
            if r[0]<s.mod_dropout_clin: fc=fc*0
            if r[1]<s.mod_dropout_clin: fb=fb*0

        st=torch.stack([fi,fc,fb],1)
        logits=s.gate_lin(torch.cat([fi,fc,fb],1))
        temp=torch.exp(s.log_temp).clamp(0.2,5.0)
        w=F.softmax(logits/temp,dim=1)

        fused=(st*w.unsqueeze(-1)).sum(1); out=s.clf(fused)

        if return_aux:
            aux_out = s.aux_mri(fi) # Train MRI projection using task labels directly
            return out, aux_out, w

        return (out,w) if rg else out

# Initializing with the gentle 0.2 dropout
model=TriModalForcedGate(encoder,CLIN_DIM,BIO_DIM,NUM_CLASSES, mod_dropout_clin=0.2).to(device)

@torch.no_grad()
def extract(loader):
    model.eval(); X=[];Y=[]
    for m,_,_,y in tqdm(loader,desc='extract'): X.append(model.imgfeat(m.to(device)).cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(X),np.concatenate(Y)
Xtr,ytr=extract(tr); Xte,yte=extract(te)
sc=StandardScaler().fit(Xtr); clf=LogisticRegression(max_iter=2000,class_weight='balanced').fit(sc.transform(Xtr),ytr)
probe_acc=accuracy_score(yte,clf.predict(sc.transform(Xte)))
Fn=Xte/(np.linalg.norm(Xte,axis=1,keepdims=True)+1e-8); off=(Fn@Fn.T)[~np.eye(len(Xte),dtype=bool)]
probe={'probe_img_only_acc':float(probe_acc),'img_sim_mean':float(off.mean()),'img_sim_std':float(off.std())}
print(f'PROBE: acc={probe_acc:.4f}  sim_std={off.std():.4f}')

extract:   0%|          | 0/17 [00:00<?, ?it/s]

extract:   0%|          | 0/4 [00:00<?, ?it/s]

PROBE: acc=0.4138  sim_std=0.3595


Cell 6: Train (With Combined Loss)

In [19]:
# train
cw=compute_class_weight('balanced',classes=np.unique(y_tr),y=y_tr)
crit=nn.CrossEntropyLoss(weight=torch.tensor(cw,dtype=torch.float).to(device))
opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=1e-4,weight_decay=1e-5)
sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=50)
best=-1; bs=None; wait=0; hist=[]

# GENTLE NUDGE: Reduced from 0.5 to 0.1
AUX_WEIGHT = 0.1

for ep in range(50):
    model.train()
    for m,c,b,y in tqdm(tr,desc=f'Ep{ep+1}',leave=False):
        m,c,b,y=m.to(device),c.to(device),b.to(device),y.to(device)
        opt.zero_grad()

        # Request auxiliary output
        main_out, aux_out, _ = model(m,c,b, return_aux=True)

        loss_main = crit(main_out, y)
        loss_aux = crit(aux_out, y)
        loss = loss_main + (AUX_WEIGHT * loss_aux) # Combined optimization

        loss.backward(); opt.step()

    model.eval(); P=[];T=[]
    with torch.no_grad():
        # During eval, return_aux defaults to False, returning standard output
        for m,c,b,y in va: P+=model(m.to(device),c.to(device),b.to(device)).argmax(1).cpu().tolist(); T+=y.tolist()
    vm=matthews_corrcoef(T,P); hist.append({'epoch':ep+1,'val_mcc':vm})

    if vm>best: best=vm; bs={k:v.cpu().clone() for k,v in model.state_dict().items()}; torch.save(bs,os.path.join(COND_DIR,'best_model.pth')); wait=0
    else:
        wait+=1
        if wait>=12: break
    sched.step()
if bs: model.load_state_dict(bs)
json.dump(hist,open(os.path.join(COND_DIR,'training_history.json'),'w'),indent=2)
print(f'trained, best val_mcc={best:.4f}')

Ep1:   0%|          | 0/17 [00:00<?, ?it/s]

Ep2:   0%|          | 0/17 [00:00<?, ?it/s]

Ep3:   0%|          | 0/17 [00:00<?, ?it/s]

Ep4:   0%|          | 0/17 [00:00<?, ?it/s]

Ep5:   0%|          | 0/17 [00:00<?, ?it/s]

...

Ep47:   0%|          | 0/17 [00:00<?, ?it/s]

Ep48:   0%|          | 0/17 [00:00<?, ?it/s]

Ep49:   0%|          | 0/17 [00:00<?, ?it/s]

Ep50:   0%|          | 0/17 [00:00<?, ?it/s]

trained, best val_mcc=0.6648


In [20]:
# diagnostic + save everything
def gm(cm):
    g=[]
    for i in range(cm.shape[0]):
        tp=cm[i,i];fn=cm[i,:].sum()-tp;fp=cm[:,i].sum()-tp;tn=cm.sum()-(tp+fn+fp)
        se=tp/(tp+fn) if tp+fn>0 else 0; sp=tn/(tn+fp) if tn+fp>0 else 0; g.append((se*sp)**0.5)
    return float(np.mean(g))

model.eval(); gates=[]; imf=[]
with torch.no_grad():
    for m,c,b,y in te:
        m,c,b=m.to(device),c.to(device),b.to(device)
        _,w=model(m,c,b,rg=True); gates.append(w.cpu().numpy()); imf.append(model.imgfeat(m).cpu().numpy())
gates=np.concatenate(gates); mg=gates.mean(0)

def ab(z=None):
    P=[];T=[]
    with torch.no_grad():
        for m,c,b,y in te:
            m,c,b=m.to(device),c.to(device),b.to(device)
            if z=='mri':m=torch.zeros_like(m)
            if z=='clin':c=torch.zeros_like(c)
            if z=='bio':b=torch.zeros_like(b)
            P+=model(m,c,b).argmax(1).cpu().tolist(); T+=y.tolist()
    cm=confusion_matrix(T,P,labels=[0,1,2]); return accuracy_score(T,P),matthews_corrcoef(T,P),gm(cm)

abl={t:dict(zip(['acc','mcc','gmean'],ab(z))) for t,z in [('full',None),('mri','mri'),('clin','clin'),('bio','bio')]}
F=np.concatenate(imf); Fn=F/(np.linalg.norm(F,axis=1,keepdims=True)+1e-8); off=(Fn@Fn.T)[~np.eye(len(F),dtype=bool)]
diag={'gate':{'mri':float(mg[0]),'clin':float(mg[1]),'bio':float(mg[2])},'ablation':abl,'img_sim_mean':float(off.mean()),'img_sim_std':float(off.std())}

print('GATE:',diag['gate']); print('ABLATION:',{k:round(v['acc'],4) for k,v in abl.items()}); print('sim_std:',round(diag['img_sim_std'],4))
full=abl['full']; mri0=abl['mri']
learned_temp=float(torch.exp(model.log_temp).clamp(0.2,5.0).item()); print('learned gate temperature:',round(learned_temp,3))

beats_3b = diag['gate']['mri']>0.141
entry={'timestamp':datetime.datetime.now(datetime.UTC).isoformat(),'condition':CONDITION_NAME,
  'change':f'{ENCODER_TO_EVAL} frozen + GENTLE GATE (asym clin dropout 0.2, aux loss 0.1); bs={BATCH_SIZE}',
  'best_val_mcc':float(best),'probe':probe,'diagnostic':diag,
  'beats_3b_ct_encoder':bool(beats_3b),'learned_temp':learned_temp,
  'baseline_ref':{'cond3b_gate_mri':0.141,'cond3b_full_acc':0.9310,'cond5_mae_gate_mri':0.1308,'cond5_mae_full_acc':0.8970}}

with open(LAB_LOG,'a') as f: f.write(json.dumps(entry)+'\n')
json.dump(entry,open(os.path.join(COND_DIR,'result.json'),'w'),indent=2)
np.save(os.path.join(COND_DIR,'test_image_features.npy'),F); np.save(os.path.join(COND_DIR,'test_gate_weights.npy'),gates)

print(f'\nvs 3b CT-encoder (gate 0.141): your {ENCODER_TO_EVAL} SSL {"BEATS" if beats_3b else "does not beat"} it')
print('All artifacts saved to', COND_DIR)

GATE: {'mri': 0.31675103306770325, 'clin': 0.6009128093719482, 'bio': 0.08233623206615448}
ABLATION: {'full': 0.7931, 'mri': 0.7241, 'clin': 0.5172, 'bio': 0.7931}
sim_std: 0.3681
learned gate temperature: 0.994

vs 3b CT-encoder (gate 0.141): your mae SSL BEATS it
All artifacts saved to <DRIVE_ROOT>/ADNI_NewDS/Phase2_Collapse_Study/results/condition6b_gentle_forced_mae
